# Imports


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import struct  # type: ignore[reportUnusedImport]
import sys  # type: ignore[reportUnusedImport]
import zipfile  # type: ignore[reportUnusedImport]
from collections import namedtuple  # type: ignore[reportUnusedImport]
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor  # type: ignore[reportUnusedImport]
from dataclasses import dataclass, field  # type: ignore[reportUnusedImport]
from datetime import datetime  # type: ignore[reportUnusedImport]
from functools import partial  # type: ignore[reportUnusedImport]
from multiprocessing import Pool  # type: ignore[reportUnusedImport]
from pathlib import Path  # type: ignore[reportUnusedImport]
from typing import List, Optional  # type: ignore[reportUnusedImport]

import geopandas as gpd  # type: ignore[reportUnusedImport]
import ggpymanager as ggp
import hvplot.xarray  # type: ignore[reportUnusedImport]
import matplotlib as mpl  # type: ignore[reportUnusedImport]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydeck as pdk  # type: ignore[reportUnusedImport]
import pyproj  # type: ignore[reportUnusedImport]
import rioxarray  # type: ignore[reportUnusedImport]
import rioxarray as rio  # type: ignore[reportUnusedImport]
import shapely  # type: ignore[reportUnusedImport]
import xarray as xr
from compliance_checker.runner import CheckSuite, ComplianceChecker  # type: ignore[reportUnusedImport]
from dask.diagnostics.progress import ProgressBar  # type: ignore[reportUnusedImport]
from joblib import Parallel, delayed  # type: ignore[reportUnusedImport]
from mpl_toolkits.axes_grid1.inset_locator import inset_axes  # type: ignore[reportUnusedImport]
from rasterio.enums import Resampling  # type: ignore[reportUnusedImport]
from tqdm import tqdm  # type: ignore[reportUnusedImport]
from windrose import WindroseAxes  # type: ignore[reportUnusedImport]

import paris_2025 as p  # type: ignore[reportUnusedImport]
from paris_2025.config import CONFIG
from paris_2025.plotting import RC_PARAMS

Using tracer path: /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers


In [3]:
# Get the root logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

plt.rcParams.update(RC_PARAMS)

# Fluxes


In [4]:
cadastre_path: str | Path = Path(CONFIG["domain"]["gral"]["conf_path"]) / "cadastre.dat"
source_groups_path: str | Path = p.model_input.fluxes.SOURCE_GROUP_NETCDF_PATH
point_path: str | Path = Path(CONFIG["domain"]["gral"]["conf_path"]) / "point.dat"
cadastre_emissions, source_groups, point_da, GRAL = (
    p.plotting._loaders.load_flux_maps_data(
        cadastre_path, source_groups_path, point_path
    )
)

In [5]:
point_fluxes = point_da.groupby("type").sum()
area_fluxes = cadastre_emissions.sum(["x", "y"]).groupby("type").sum()
df = pd.concat(
    [area_fluxes.to_pandas(), point_fluxes.to_pandas()],  # type: ignore
    axis=1,
    keys=["area", "point"],
)
# Convert from kg/h to kt/year
df = df * 365 * 24 / 1e6
df.round(0)

,area,point
type,,
Origins.earth 2023 energie,308.0,751.0
Origins.earth 2023 industrie,582.0,60.0
Origins.earth 2023 residentiel,1894.0,NaN
Origins.earth 2023 respiration_humaine,965.0,NaN
Origins.earth 2023 tertiaire,466.0,NaN
Origins.earth 2023 transport_routier,2707.0,NaN
TNO 2018 Combustion,4696.0,NaN
TNO 2018 Industry,205.0,NaN
TNO 2018 Power,135.0,2481.0


In [6]:
inventories = ["Origins.earth", "TNO"]
totals = pd.DataFrame(
    {inv: df[df.index.str.contains(inv)].sum() for inv in inventories}
).T
totals["total"] = totals.sum(axis=1)

relative_contributions = totals.div(totals["total"], axis=0) * 100

display(totals.round(0))
display(relative_contributions.round(1))

,area,point,total
Origins.earth,6921.0,811.0,7732.0
TNO,6950.0,2481.0,9431.0


,area,point,total
Origins.earth,89.5,10.5,100.0
TNO,73.7,26.3,100.0


# CO2 Measurements


In [7]:
co2 = ggp.load("co2_measurements", CONFIG).co2

# Calculate the available measurements in 2023 to 2024
co2_period = co2.sel(time=slice("2023", "2024"))
n_total = co2_period.sizes["time"]
counts = co2_period.count(dim="time")
co2["measurement_availability"] = (
    counts.astype(str) + " (" + (counts / n_total * 100).round(1).astype(str) + "%)"
)

# Convert to a DataFrame and select relevant columns
df = (
    co2.coords.to_dataset()
    .drop_vars("time")
    .to_pandas()[
        [
            "height",
            "latitude",
            "longitude",
            "code",
            "instrument",
            "in_gral_domain",
            "measurement_availability",
        ]
    ]
)
# Rename columns
df = df.rename(
    columns={
        "station": "Station",
        "height": "Height (m a.s.l.)",
        "latitude": "Latitude",
        "longitude": "Longitude",
        "code": "Code",
        "instrument": "Instrument",
        "in_gral_domain": "In GRAL Domain",
        "measurement_availability": "Number of Measurements (Percentage)",
    }
)
# Sort by Instrument, then by Code, and then by Height
df = df.sort_values(
    by=["Instrument", "Code", "Height (m a.s.l.)"],
    ascending=[False, True, True],
)

# Drop "Code"
df = df.drop(columns=["Code"])

# Escape underscores in the index for LaTeX
df.index = df.index.str.replace("_", "\\_", regex=False)

# Escape the percentage sign in the "Number of Measurements (Percentage)" column for LaTeX
df["Number of Measurements (Percentage)"] = df[
    "Number of Measurements (Percentage)"
].str.replace("%", "\\%", regex=False)

# Round height to the nearest meter
df["Height (m a.s.l.)"] = df["Height (m a.s.l.)"].round().astype(int)

# Round latitude and longitude to 4 decimal places
df["Latitude"] = df["Latitude"].round(4)
df["Longitude"] = df["Longitude"].round(4)


print(df.to_latex(float_format="%.4f"))
df

INFO:root:Opening co2_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measurements/6_2_tracers/co2.nc


\begin{tabular}{lrrrlrl}
\toprule
 & Height (m a.s.l.) & Latitude & Longitude & Instrument & In GRAL Domain & Number of Measurements (Percentage) \\
station &  &  &  &  &  &  \\
\midrule
AND\_60 & 60 & 49.0126 & 2.3018 & Picarro & False & 17241 (98.3\%) \\
CDS\_34 & 34 & 48.8956 & 2.3880 & Picarro & True & 16272 (92.7\%) \\
COU\_30 & 30 & 48.9242 & 2.5680 & Picarro & False & 14527 (82.8\%) \\
GNS\_2 & 2 & 49.0052 & 2.4205 & Picarro & False & 12210 (69.6\%) \\
GNS\_36 & 36 & 49.0052 & 2.4205 & Picarro & False & 12830 (73.1\%) \\
JUS\_30 & 30 & 48.8464 & 2.3561 & Picarro & True & 14121 (80.5\%) \\
JUS\_40 & 40 & 48.8464 & 2.3561 & Picarro & True & 9758 (55.6\%) \\
MEU\_45 & 45 & 48.8025 & 2.2044 & Picarro & True & 11088 (63.2\%) \\
MEU\_65 & 65 & 48.8025 & 2.2044 & Picarro & True & 11101 (63.3\%) \\
MEU\_90 & 90 & 48.8025 & 2.2044 & Picarro & True & 16491 (94.0\%) \\
OVS\_20 & 20 & 48.7779 & 2.0486 & Picarro & False & 16595 (94.6\%) \\
ROV\_103 & 103 & 48.8854 & 2.4225 & Picarro & True &

,Height (m a.s.l.),Latitude,Longitude,Instrument,In GRAL Domain,Number of Measurements (Percentage)
station,,,,,,
AND\_60,60,49.0126,2.3018,Picarro,False,17241 (98.3\%)
CDS\_34,34,48.8956,2.3880,Picarro,True,16272 (92.7\%)
COU\_30,30,48.9242,2.5680,Picarro,False,14527 (82.8\%)
GNS\_2,2,49.0052,2.4205,Picarro,False,12210 (69.6\%)
GNS\_36,36,49.0052,2.4205,Picarro,False,12830 (73.1\%)
JUS\_30,30,48.8464,2.3561,Picarro,True,14121 (80.5\%)
JUS\_40,40,48.8464,2.3561,Picarro,True,9758 (55.6\%)
MEU\_45,45,48.8025,2.2044,Picarro,True,11088 (63.2\%)
MEU\_65,65,48.8025,2.2044,Picarro,True,11101 (63.3\%)


# Ensemble size


In [8]:
conc_series = ggp.load("concentration_timeseries", CONFIG)
loss_diff = conc_series["loss_diff"].sel(loss_type="rmse - filter: True")
mask = (loss_diff < 0.1).sum("best_sim_id")
print(
    "Take the 95% percentile as a threshold for the number of simulations that are "
    "below the 0.1 RMSE threshold."
)
display(mask.to_pandas().describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))
limit = int((0.9 * len(mask)))
print(f"Select top 10% of the simulations: {len(mask) - limit}/{len(mask)}")
mean = mask.sortby(mask).values[limit:].mean()
print(
    f"Mean number of simulations below the 0.1 RMSE threshold for the top 10%: "
    f"{mean:.2f}"
)

INFO:root:Opening concentration_timeseries from /Users/rmaiwald/Levante/Paris/Output/concentration_timeseries.nc


Take the 95% percentile as a threshold for the number of simulations that are below the 0.1 RMSE threshold.


count    17544.000000
mean         3.838862
std          4.495322
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
90%          7.000000
95%         10.000000
99%         23.000000
max         89.000000
Name: loss_diff, dtype: float64

Select top 10% of the simulations: 1755/17544
Mean number of simulations below the 0.1 RMSE threshold for the top 10%: 13.28


# Wind Measurements


In [9]:
model_meteo_timeseries = (
    ggp.load("model_meteo_timeseries", CONFIG)
    .sel(loss_type="rmse - filter: True")
    .load()
)
meteo = ggp.load("meteo_measurements", CONFIG)
meteo = meteo.sel(
    station=model_meteo_timeseries.station, time=model_meteo_timeseries.time
).load()

INFO:root:Opening gramm_meteo_timeseries from /Users/rmaiwald/Levante/Paris/Output/gramm_meteo_timeseries.nc
INFO:root:Opening gral_meteo_timeseries from /Users/rmaiwald/Levante/Paris/Output/gral_meteo_timeseries.nc
INFO:root:Selecting model data for station LONGCHAMP from model gral
INFO:root:Selecting model data for station PARIS-MONTSOURIS from model gral
INFO:root:Selecting model data for station TOUR EIFFEL from model gramm
INFO:root:Selecting model data for station LFPB from model gramm
INFO:root:Selecting model data for station LFPN_2 from model gramm
INFO:root:Selecting model data for station LFPO from model gramm
INFO:root:Selecting model data for station LFPV from model gramm
INFO:root:Selecting model data for station Cité des Sciences from model gral
INFO:root:Selecting model data for station Meudon from model gramm
INFO:root:Selecting model data for station Romainville from model gral
INFO:root:Opening meteo_measurements from /Users/rmaiwald/Levante/Paris/Input/6_measuremen

In [10]:
wind_speed_diff = model_meteo_timeseries["wind_speed"] - meteo["wind_speed"]
wind_direction_diff = ggp.processing.circular_diff(
    model_meteo_timeseries["wind_direction"], meteo["wind_direction"]
)
u_wind_diff = model_meteo_timeseries["u"] - meteo["u_wind"]
v_wind_diff = model_meteo_timeseries["v"] - meteo["v_wind"]

In [11]:
def model_performance_df(
    model: xr.Dataset,
    meteo: xr.Dataset,
) -> pd.DataFrame:
    """Create a DataFrame of model performance metrics from wind model and measurements.

    Computes bias, MAE, RMSE, and Pearson R per station averaged over time and
    (if present) all ensemble members (``best_sim_id`` dimension). u and v are
    combined into a single wind vector variable: bias is reported as separate u/v
    columns, MAE and RMSE are computed on the vector magnitude (no R for the vector).
    An additional column reports the mean observed wind speed per station.

    Parameters
    ----------
    model : xr.Dataset
        Model output with variables ``wind_speed``, ``wind_direction``, ``u``, ``v``.
    meteo : xr.Dataset
        Measurements with variables ``wind_speed``, ``wind_direction``,
        ``u_wind``, ``v_wind``.

    Returns
    -------
    pd.DataFrame
        Multi-level column DataFrame indexed by station with columns
        ``(variable, metric)``, plus a final "Mean" row.
    """

    def _avg_dims(da: xr.DataArray) -> list[str]:
        return [d for d in da.dims if d != "station"]

    def _metrics(model_var: xr.DataArray, meas_var: xr.DataArray) -> pd.DataFrame:
        diff: xr.DataArray = (model_var - meas_var).astype(float)  # type: ignore[assignment]
        avg_dims = _avg_dims(diff)
        bias = diff.mean(dim=avg_dims)
        mae = abs(diff).mean(dim=avg_dims)
        rmse = (diff**2).mean(dim=avg_dims) ** 0.5
        r = xr.corr(model_var.astype(float), meas_var.astype(float), dim=avg_dims)  # type: ignore[arg-type]
        return pd.DataFrame(
            {
                "Bias (m/s)": bias.to_pandas(),
                # "MAE": mae.to_pandas(),
                "RMSE (m/s)": rmse.to_pandas(),
                "R": r.to_pandas(),
            }
        )

    def _vector_metrics(
        u_model: xr.DataArray,
        v_model: xr.DataArray,
        u_meas: xr.DataArray,
        v_meas: xr.DataArray,
    ) -> pd.DataFrame:
        fu: xr.DataArray = (u_model - u_meas).astype(float)  # type: ignore[assignment]
        fv: xr.DataArray = (v_model - v_meas).astype(float)  # type: ignore[assignment]
        avg_dims = _avg_dims(fu)
        u_bias = fu.mean(dim=avg_dims)
        v_bias = fv.mean(dim=avg_dims)
        mae = ((fu**2 + fv**2) ** 0.5).mean(dim=avg_dims)
        rmse = ((fu**2 + fv**2).mean(dim=avg_dims)) ** 0.5
        return pd.DataFrame(
            {
                # "Bias (u)": u_bias.to_pandas(),
                # "Bias (v)": v_bias.to_pandas(),
                # "MAE": mae.to_pandas(),
                "RMSE (m/s)": rmse.to_pandas(),
            }
        )

    wind_direction_diff = ggp.processing.circular_diff(
        model["wind_direction"], meteo["wind_direction"]
    )
    wind_dir_corr = xr.corr(
        model["wind_direction"].astype(float),  # type: ignore[arg-type]
        meteo["wind_direction"].astype(float),  # type: ignore[arg-type]
        dim=_avg_dims(wind_direction_diff),
    )

    obs_speed = (
        meteo["wind_speed"]
        .astype(float)
        .mean(dim=_avg_dims(meteo["wind_speed"]))  # type: ignore[assignment]
    )

    frames: dict[str, pd.DataFrame] = {
        "Obs. speed": pd.DataFrame({"(m/s)": obs_speed.to_pandas()}),
        "Vector": _vector_metrics(
            model["u"], model["v"], meteo["u_wind"], meteo["v_wind"]
        ),
        "Speed": _metrics(model["wind_speed"], meteo["wind_speed"]),
        "Dir. (°)": pd.DataFrame(
            {
                # "Bias": wind_direction_diff.astype(float).mean(dim=_avg_dims(wind_direction_diff)).to_pandas(),  # type: ignore[assignment]
                # "MAE": abs(wind_direction_diff.astype(float)).mean(dim=_avg_dims(wind_direction_diff)).to_pandas(),  # type: ignore[assignment]
                "RMSE": ((wind_direction_diff.astype(float) ** 2).mean(dim=_avg_dims(wind_direction_diff)) ** 0.5).to_pandas(),  # type: ignore[assignment]
                # "R": wind_dir_corr.to_pandas(),
            }
        ),
    }

    df = pd.concat(frames, axis=1)

    mean_row = df.mean().rename("Mean").to_frame().T
    df = pd.concat([df, mean_row])
    df.index.name = "Station"
    return df


df = model_performance_df(model_meteo_timeseries, meteo).round(2)

# Replace all underscores in the index with escaped underscores for LaTeX
df.index = df.index.str.replace("_", "\\_", regex=False)

In [12]:
latex = df.to_latex(float_format="%.2f")

# Insert \midrule before the last data row and make it bold
lines = latex.splitlines()
# Find the last data line (before \bottomrule)
for i in range(len(lines) - 1, -1, -1):
    if lines[i].strip() == r"\bottomrule":
        # Insert \midrule before the last data row
        last_data_idx = i - 1
        cells = lines[last_data_idx].rstrip(r" \\").split(" & ")
        lines[last_data_idx] = (
            " & ".join(f"\\textbf{{{c.strip()}}}" for c in cells) + r" \\"
        )
        lines.insert(last_data_idx, r"\midrule")
        break

print("\n".join(lines))
display(df.style.background_gradient(axis="index").format("{:.2f}"))

\begin{tabular}{lrrrrrr}
\toprule
 & Obs. speed & Vector & \multicolumn{3}{r}{Speed} & Dir. (°) \\
 & (m/s) & RMSE (m/s) & Bias (m/s) & RMSE (m/s) & R & RMSE \\
Station &  &  &  &  &  &  \\
\midrule
LONGCHAMP & 2.51 & 1.42 & 0.13 & 1.04 & 0.77 & 45.01 \\
PARIS-MONTSOURIS & 3.11 & 1.90 & -1.07 & 1.41 & 0.76 & 36.30 \\
TOUR EIFFEL & 7.20 & 4.14 & -0.85 & 2.96 & 0.68 & 32.84 \\
LFPB & 3.59 & 2.29 & -0.86 & 1.58 & 0.76 & 49.53 \\
LFPN\_2 & 3.83 & 2.34 & -0.82 & 1.57 & 0.80 & 45.45 \\
LFPO & 3.73 & 2.22 & -0.68 & 1.47 & 0.76 & 42.66 \\
LFPV & 3.71 & 2.32 & -1.11 & 1.64 & 0.74 & 45.49 \\
Cité des Sciences & 2.64 & 1.30 & 0.48 & 1.02 & 0.83 & 23.54 \\
Meudon & 4.72 & 4.22 & -1.06 & 1.90 & 0.72 & 55.85 \\
Romainville & 6.03 & 1.93 & -0.68 & 1.55 & 0.88 & 14.15 \\
\midrule
\textbf{Mean} & \textbf{4.11} & \textbf{2.41} & \textbf{-0.65} & \textbf{1.61} & \textbf{0.77} & \textbf{39.08} \\
\bottomrule
\end{tabular}
